In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.random.seed(42)

In [3]:
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Target names: {data.target_names}")
print(f"Feature names (first 10): {data.feature_names[:10]}")
print(f"Class distribution - Malignant (0): {np.sum(y == 0)}, Benign (1): {np.sum(y == 1)}")

Dataset shape: (569, 30)
Target names: ['malignant' 'benign']
Feature names (first 10): ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension']
Class distribution - Malignant (0): 212, Benign (1): 357


In [4]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

random_states = [10, 20, 30, 40]
accuracy_results = []

for rs in random_states:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=rs, stratify=y
    )
    
    lr_pipeline.fit(X_train, y_train)
    y_pred = lr_pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracy_results.append(acc)
    print(f"{rs:<15} {acc:.6f}")

print(f"Mean accuracy across splits: {np.mean(accuracy_results):.6f}")
print(f"Standard deviation across splits: {np.std(accuracy_results):.6f}")

10              0.964912
20              0.982456
30              0.982456
40              1.000000
Mean accuracy across splits: 0.982456
Standard deviation across splits: 0.012405


In [5]:
scores = cross_val_score(
    lr_pipeline, X, y, cv=5, scoring='accuracy'
)

print(f"\nFold scores: {scores}")
print(f"Mean accuracy: {scores.mean():.6f}")
print(f"Standard deviation: {scores.std():.6f}")


Fold scores: [0.98245614 0.98245614 0.97368421 0.97368421 0.99115044]
Mean accuracy: 0.980686
Standard deviation: 0.006539


In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores_stratified = cross_val_score(lr_pipeline, X, y, cv=cv, scoring='accuracy')

print(f"Fold scores: {scores_stratified}")
print(f"Mean accuracy: {scores_stratified.mean():.6f}")
print(f"Standard deviation: {scores_stratified.std():.6f}")

Fold scores: [0.97368421 0.94736842 0.96491228 0.99122807 0.99115044]
Mean accuracy: 0.973669
Standard deviation: 0.016627


In [7]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVC Pipeline': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf'))
    ])
}

results = {}

print(f"{'Model':<25}\t{'Mean Accuracy':<15}\t{'Std Dev':<15}")

for name, clf in models.items():
    scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
    results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores}
    print(f"{name:<25}\t{scores.mean():.6f}\t{scores.std():.6f}")

Model                    	Mean Accuracy  	Std Dev        
Logistic Regression      	0.973669	0.016627
KNN                      	0.963096	0.017886
Decision Tree            	0.910402	0.027876
SVC Pipeline             	0.977162	0.016260


In [8]:
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Mean Accuracy': [results[m]['mean'] for m in results.keys()],
    'Standard Deviation': [results[m]['std'] for m in results.keys()]
})
comparison_df = comparison_df.sort_values('Mean Accuracy', ascending=False)

print("\n" + comparison_df.to_string(index=False))

models_names = list(results.keys())
means = [results[m]['mean'] for m in models_names]
stds = [results[m]['std'] for m in models_names]


              Model  Mean Accuracy  Standard Deviation
       SVC Pipeline       0.977162            0.016260
Logistic Regression       0.973669            0.016627
                KNN       0.963096            0.017886
      Decision Tree       0.910402            0.027876


In [9]:
best_model = max(results.keys(), key=lambda x: results[x]['mean'])
most_stable = min(results.keys(), key=lambda x: results[x]['std'])

print(f"Best performing model (highest mean accuracy): {best_model}")
print(f"\tMean Accuracy: {results[best_model]['mean']:.6f}")
print(f"\tStd Dev: {results[best_model]['std']:.6f}")

print(f"\nMost stable model (lowest standard deviation): {most_stable}")
print(f"\tMean Accuracy: {results[most_stable]['mean']:.6f}")
print(f"\tStd Dev: {results[most_stable]['std']:.6f}")

for name in models_names:
    mean_acc = results[name]['mean']
    std_acc = results[name]['std']
    print(f"\n{name}:")
    print(f"\tMean Accuracy: {mean_acc:.6f}")
    print(f"\tStandard Deviation: {std_acc:.6f}")

Best performing model (highest mean accuracy): SVC Pipeline
	Mean Accuracy: 0.977162
	Std Dev: 0.016260

Most stable model (lowest standard deviation): SVC Pipeline
	Mean Accuracy: 0.977162
	Std Dev: 0.016260

Logistic Regression:
	Mean Accuracy: 0.973669
	Standard Deviation: 0.016627

KNN:
	Mean Accuracy: 0.963096
	Standard Deviation: 0.017886

Decision Tree:
	Mean Accuracy: 0.910402
	Standard Deviation: 0.027876

SVC Pipeline:
	Mean Accuracy: 0.977162
	Standard Deviation: 0.016260
